# SST Next-Brick Pose Prediction

Small Transformer encoder + MDN for next brick pose prediction from partial assembly history.

**Architecture:** 2-layer Transformer encoder, CLS token -> binary-state head + MDN head (K=5)  
**Data:** batch2 and batch3 validated_simPhysics sequences

In [ ]:
import json
import os
import math
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

# Run from project root regardless of Jupyter launch directory
cwd = Path(os.getcwd())
if cwd.name == "training_data":
    os.chdir(cwd.parent)
print(f"Working directory: {os.getcwd()}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}  |  PyTorch: {torch.__version__}")

: 

In [ ]:
# Data
BATCH_DIRS = [
    "training_data/batch2/validated_simPhysics",
    "training_data/batch3/validated_simPhysics",
]
SEQ_SUBPATH = "5d_sequence/sequence.json"

# Model
FEATURE_DIM = 8
HIDDEN_DIM  = 128
N_HEADS     = 4
N_LAYERS    = 2
FF_DIM      = 256
DROPOUT     = 0.1
K_MIXTURES  = 5
MAX_BRICKS  = 60

# Training
BATCH_SIZE   = 64
LR           = 3e-4
WEIGHT_DECAY = 1e-4
MAX_EPOCHS   = 150
PATIENCE     = 20
GRAD_CLIP    = 1.0
LAMBDA_B     = 1.0
N_AUG        = 50

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

## 1. Data Loading

In [ ]:
def load_sequences(batch_dirs, seq_subpath=SEQ_SUBPATH):
    sequences = []
    for bd in batch_dirs:
        bd_path = Path(bd)
        if not bd_path.exists():
            print(f"Warning: {bd_path} not found, skipping.")
            continue
        for demo_dir in sorted(bd_path.iterdir()):
            seq_path = demo_dir / seq_subpath
            if seq_path.exists():
                with open(seq_path) as f:
                    poses = json.load(f)
                sequences.append({"id": str(demo_dir), "poses": poses})
    return sequences


sequences = load_sequences(BATCH_DIRS)
print(f"Loaded {len(sequences)} sequences:\n")
for s in sequences:
    print(f"  {s['id']}  ->  {len(s['poses'])} bricks")

## 2. Feature Engineering

In [ ]:
def assign_layer_ids(poses, z_tol=1e-4):
    """Group bricks into layers by clustering similar z values."""
    unique_z = []
    for p in poses:
        z = p[2]
        if not any(abs(z - uz) < z_tol for uz in unique_z):
            unique_z.append(z)
    unique_z.sort()
    return [
        min(range(len(unique_z)), key=lambda i: abs(unique_z[i] - p[2]))
        for p in poses
    ]


def canonicalize_r(r):
    """Map yaw r to [0, pi) exploiting brick 180-degree symmetry."""
    r = r % math.pi
    return r + math.pi if r < 0 else r


def encode_brick(pose, layer_id, time_index):
    """Return 8-dim feature vector: [x, y, z, b, sin_r, cos_r, layer_id, time_index]."""
    x, y, z, b, r = pose
    r_c = canonicalize_r(r)
    return [x, y, z, b, math.sin(r_c), math.cos(r_c), float(layer_id), float(time_index)]

## 3. Sequence -> Next-Step Training Pairs

In [ ]:
def sequence_to_pairs(seq_data):
    """Convert one full sequence to T next-step (history -> target) pairs."""
    poses     = seq_data["poses"]
    layer_ids = assign_layer_ids(poses)
    pairs = []
    for t in range(len(poses)):
        history_encoded = [
            encode_brick(poses[i], layer_ids[i], i) for i in range(t)
        ]
        x, y, z, b, r = poses[t]
        r_c = canonicalize_r(r)
        target = {
            "x": x, "y": y, "z": z,
            "b": int(b),
            "sin_r": math.sin(r_c),
            "cos_r": math.cos(r_c),
            "layer_id": layer_ids[t],
        }
        pairs.append({"history": history_encoded, "target": target, "seq_id": seq_data["id"]})
    return pairs

## 4. Sequence-Level Train / Val / Test Split

In [ ]:
indices = list(range(len(sequences)))
random.shuffle(indices)

n_total = len(sequences)
n_val   = max(1, round(0.15 * n_total))
n_test  = max(1, round(0.15 * n_total))
n_train = n_total - n_val - n_test

train_idx = indices[:n_train]
val_idx   = indices[n_train : n_train + n_val]
test_idx  = indices[n_train + n_val :]

train_seqs = [sequences[i] for i in train_idx]
val_seqs   = [sequences[i] for i in val_idx]
test_seqs  = [sequences[i] for i in test_idx]

print(f"Split: {n_train} train / {n_val} val / {n_test} test  (total {n_total})")
print(f"Train: {[Path(sequences[i]['id']).name for i in train_idx]}")
print(f"Val:   {[Path(sequences[i]['id']).name for i in val_idx]}")
print(f"Test:  {[Path(sequences[i]['id']).name for i in test_idx]}")

train_pairs_orig = [p for seq in train_seqs for p in sequence_to_pairs(seq)]
val_pairs        = [p for seq in val_seqs   for p in sequence_to_pairs(seq)]
test_pairs       = [p for seq in test_seqs  for p in sequence_to_pairs(seq)]

print(f"\nPairs before aug -- train: {len(train_pairs_orig)}, val: {len(val_pairs)}, test: {len(test_pairs)}")

## 5. SE(2) Data Augmentation  (training only)

In [ ]:
def _rotate_xy(x, y, cx, cy, cos_t, sin_t, tx, ty):
    dx, dy = x - cx, y - cy
    return cx + cos_t * dx - sin_t * dy + tx, cy + sin_t * dx + cos_t * dy + ty


def _rotate_sincos(sin_r, cos_r, theta):
    r_new = canonicalize_r(math.atan2(sin_r, cos_r) + theta)
    return math.sin(r_new), math.cos(r_new)


def se2_augment(pair, theta, tx, ty):
    """Apply a rigid SE(2) transform centred on the sample bounding-box."""
    cos_t, sin_t = math.cos(theta), math.sin(theta)
    all_x = [h[0] for h in pair["history"]] + [pair["target"]["x"]]
    all_y = [h[1] for h in pair["history"]] + [pair["target"]["y"]]
    cx, cy = sum(all_x) / len(all_x), sum(all_y) / len(all_y)

    new_history = []
    for h in pair["history"]:
        x, y, z, b, sin_r, cos_r, lid, tidx = h
        xn, yn = _rotate_xy(x, y, cx, cy, cos_t, sin_t, tx, ty)
        sn, cn = _rotate_sincos(sin_r, cos_r, theta)
        new_history.append([xn, yn, z, b, sn, cn, lid, tidx])

    tgt = pair["target"]
    xn, yn = _rotate_xy(tgt["x"], tgt["y"], cx, cy, cos_t, sin_t, tx, ty)
    sn, cn = _rotate_sincos(tgt["sin_r"], tgt["cos_r"], theta)
    new_target = {**tgt, "x": xn, "y": yn, "sin_r": sn, "cos_r": cn}
    return {"history": new_history, "target": new_target, "seq_id": pair["seq_id"]}


def augment_pairs(pairs, n_aug):
    aug = []
    for pair in pairs:
        for _ in range(n_aug):
            theta = random.uniform(0, 2 * math.pi)
            tx    = random.uniform(-0.01, 0.01)
            ty    = random.uniform(-0.01, 0.01)
            aug.append(se2_augment(pair, theta, tx, ty))
    return aug


print(f"Generating {N_AUG} augmentations x {len(train_pairs_orig)} training pairs...")
train_pairs_aug = augment_pairs(train_pairs_orig, N_AUG)
all_train_pairs = train_pairs_orig + train_pairs_aug
print(f"Total training pairs after augmentation: {len(all_train_pairs):,}")

## 6. Normalization Statistics  (training data only)

In [ ]:
def compute_norm_stats(pairs):
    xs, ys, zs = [], [], []
    for p in pairs:
        xs.append(p["target"]["x"])
        ys.append(p["target"]["y"])
        zs.append(p["target"]["z"])
        for h in p["history"]:
            xs.append(h[0]); ys.append(h[1]); zs.append(h[2])
    return {
        "mean_x": float(np.mean(xs)), "std_x": float(np.std(xs)) + 1e-8,
        "mean_y": float(np.mean(ys)), "std_y": float(np.std(ys)) + 1e-8,
        "mean_z": float(np.mean(zs)), "std_z": float(np.std(zs)) + 1e-8,
    }


norm_stats = compute_norm_stats(all_train_pairs)
print("Normalization statistics (from training data only):")
for k, v in norm_stats.items():
    print(f"  {k:8s}: {v:.6f}")

## 7. PyTorch Dataset & DataLoaders

In [ ]:
class BrickPoseDataset(Dataset):
    def __init__(self, pairs, norm_stats, max_bricks=MAX_BRICKS):
        self.pairs = pairs
        self.ns    = norm_stats
        self.max_bricks = max_bricks

    def _norm_xyz(self, x, y, z):
        ns = self.ns
        return (
            (x - ns["mean_x"]) / ns["std_x"],
            (y - ns["mean_y"]) / ns["std_y"],
            (z - ns["mean_z"]) / ns["std_z"],
        )

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pair   = self.pairs[idx]
        tokens = torch.zeros(self.max_bricks, FEATURE_DIM, dtype=torch.float32)
        mask   = torch.zeros(self.max_bricks, dtype=torch.bool)
        for i, h in enumerate(pair["history"][:self.max_bricks]):
            x, y, z, b, sin_r, cos_r, layer_id, time_index = h
            xn, yn, zn = self._norm_xyz(x, y, z)
            tokens[i] = torch.tensor(
                [xn, yn, zn, b, sin_r, cos_r, layer_id / 20.0, time_index / 60.0]
            )
            mask[i] = True
        tgt = pair["target"]
        xn, yn, zn = self._norm_xyz(tgt["x"], tgt["y"], tgt["z"])
        target_cont = torch.tensor(
            [xn, yn, zn, tgt["sin_r"], tgt["cos_r"]], dtype=torch.float32
        )
        target_b = torch.tensor(tgt["b"], dtype=torch.long)
        return tokens, mask, target_cont, target_b


train_ds = BrickPoseDataset(all_train_pairs, norm_stats)
val_ds   = BrickPoseDataset(val_pairs,       norm_stats)
test_ds  = BrickPoseDataset(test_pairs,      norm_stats)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train: {len(train_ds):,} samples  ({len(train_loader)} batches)")
print(f"Val:   {len(val_ds):,} samples  ({len(val_loader)} batches)")
print(f"Test:  {len(test_ds):,} samples  ({len(test_loader)} batches)")

## 8. Model Architecture

Input tokens -> MLP projection -> Transformer encoder (pre-norm) -> CLS token  
Scene embedding -> binary-state head + MDN head (K=5 Gaussian components over [x, y, z, sin_r, cos_r])

In [ ]:
class NextBrickModel(nn.Module):
    """Transformer encoder + MDN head for next brick pose prediction."""

    def __init__(
        self,
        feature_dim=FEATURE_DIM, hidden_dim=HIDDEN_DIM,
        nhead=N_HEADS, num_layers=N_LAYERS, ff_dim=FF_DIM,
        dropout=DROPOUT, K=K_MIXTURES,
    ):
        super().__init__()
        self.K        = K
        self.pose_dim = 5  # [x, y, z, sin_r, cos_r]

        self.input_proj = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.cls_token = nn.Parameter(torch.randn(1, 1, hidden_dim) * 0.02)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim, nhead=nhead, dim_feedforward=ff_dim,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        self.b_head = nn.Sequential(
            nn.Linear(hidden_dim, 64), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(64, 2),
        )
        self.mdn_pi        = nn.Linear(hidden_dim, K)
        self.mdn_mu        = nn.Linear(hidden_dim, K * self.pose_dim)
        self.mdn_log_sigma = nn.Linear(hidden_dim, K * self.pose_dim)

    def forward(self, tokens, mask):
        """
        tokens : (B, T, F)  padded brick tokens
        mask   : (B, T)     True=real brick, False=padding
        """
        B = tokens.shape[0]
        x   = self.input_proj(tokens)
        cls = self.cls_token.expand(B, 1, -1)
        x   = torch.cat([cls, x], dim=1)
        cls_valid = torch.ones(B, 1, dtype=torch.bool, device=mask.device)
        full_mask = torch.cat([cls_valid, mask], dim=1)
        x = self.transformer(x, src_key_padding_mask=~full_mask)
        scene    = x[:, 0]
        b_logits = self.b_head(scene)
        pi    = F.softmax(self.mdn_pi(scene), dim=-1)
        mu    = self.mdn_mu(scene).view(B, self.K, self.pose_dim)
        sigma = F.softplus(self.mdn_log_sigma(scene)).view(B, self.K, self.pose_dim) + 1e-4
        return b_logits, pi, mu, sigma

## 9. Loss Functions

In [ ]:
_LOG2PI = math.log(2 * math.pi)


def mdn_nll_loss(pi, mu, sigma, target):
    """Gaussian MDN NLL via log-sum-exp.
    pi:(B,K)  mu:(B,K,D)  sigma:(B,K,D)  target:(B,D)
    """
    target    = target.unsqueeze(1).expand_as(mu)
    log_gauss = (
        -0.5 * (((target - mu) / sigma).pow(2) + 2 * sigma.log() + _LOG2PI)
    ).sum(-1)
    log_mix = torch.logsumexp(torch.log(pi + 1e-8) + log_gauss, dim=-1)
    return -log_mix.mean()


def compute_loss(b_logits, pi, mu, sigma, target_cont, target_b):
    l_mdn = mdn_nll_loss(pi, mu, sigma, target_cont)
    l_b   = F.cross_entropy(b_logits, target_b)
    return l_mdn + LAMBDA_B * l_b, l_mdn, l_b

## 10. Training & Validation Loops

In [ ]:
def train_epoch(model, loader, optimizer):
    model.train()
    total = 0.0
    for tokens, mask, tgt_cont, tgt_b in loader:
        tokens, mask    = tokens.to(device), mask.to(device)
        tgt_cont, tgt_b = tgt_cont.to(device), tgt_b.to(device)
        optimizer.zero_grad()
        loss, _, _ = compute_loss(*model(tokens, mask), tgt_cont, tgt_b)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def eval_epoch(model, loader):
    model.eval()
    tot, tot_mdn, tot_b = 0.0, 0.0, 0.0
    correct, n = 0, 0
    for tokens, mask, tgt_cont, tgt_b in loader:
        tokens, mask    = tokens.to(device), mask.to(device)
        tgt_cont, tgt_b = tgt_cont.to(device), tgt_b.to(device)
        b_logits, pi, mu, sigma = model(tokens, mask)
        loss, l_mdn, l_b = compute_loss(b_logits, pi, mu, sigma, tgt_cont, tgt_b)
        tot += loss.item(); tot_mdn += l_mdn.item(); tot_b += l_b.item()
        correct += (b_logits.argmax(-1) == tgt_b).sum().item()
        n += len(tgt_b)
    N = len(loader)
    return tot / N, tot_mdn / N, tot_b / N, correct / n

## 11. Main Training Loop

In [ ]:
model     = NextBrickModel().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=10
)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {n_params:,}")

best_val, no_improve = float("inf"), 0
CKPT = Path("training_data/best_model.pth")
log  = {"train": [], "val": [], "val_mdn": [], "val_b": [], "val_acc": []}

for epoch in range(1, MAX_EPOCHS + 1):
    tr                        = train_epoch(model, train_loader, optimizer)
    vl, vl_mdn, vl_b, vl_acc  = eval_epoch(model, val_loader)
    scheduler.step(vl)
    log["train"].append(tr);        log["val"].append(vl)
    log["val_mdn"].append(vl_mdn);  log["val_b"].append(vl_b)
    log["val_acc"].append(vl_acc)

    if vl < best_val:
        best_val, no_improve = vl, 0
        torch.save({
            "epoch": epoch, "model_state": model.state_dict(),
            "optimizer_state": optimizer.state_dict(),
            "norm_stats": norm_stats, "val_loss": vl,
        }, CKPT)
    else:
        no_improve += 1

    if epoch % 10 == 0 or epoch == 1:
        lr_now = optimizer.param_groups[0]["lr"]
        print(
            f"[{epoch:3d}] train={tr:.3f} | val={vl:.3f} "
            f"(mdn={vl_mdn:.3f} b={vl_b:.3f}) | b_acc={vl_acc:.3f} | lr={lr_now:.1e}"
        )
    if no_improve >= PATIENCE:
        print(f"\nEarly stop at epoch {epoch}.")
        break

print(f"\nBest val loss: {best_val:.4f}  ->  {CKPT}")

## 12. Training Curves

In [ ]:
E = range(1, len(log["train"]) + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(E, log["train"], label="train")
axes[0].plot(E, log["val"],   label="val")
axes[0].set_title("Total Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
axes[1].plot(E, log["val_mdn"], label="MDN NLL")
axes[1].plot(E, log["val_b"],   label="Binary CE")
axes[1].set_title("Val Loss Components"); axes[1].set_xlabel("Epoch"); axes[1].legend()
axes[2].plot(E, log["val_acc"])
axes[2].set_title("Val Binary-State Accuracy")
axes[2].set_xlabel("Epoch"); axes[2].set_ylim(0, 1)
axes[2].axhline(0.5, color="gray", linestyle="--", alpha=0.5, label="chance")
axes[2].legend()
plt.tight_layout()
plt.savefig("training_data/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: training_data/training_curves.png")

## 13. Inference -- Candidate Sampling

In [ ]:
def sample_next_brick(model, history_encoded, norm_stats, n_candidates=100):
    """
    Sample n_candidates next brick poses from the model.
    history_encoded : list of 8-dim vectors from encode_brick()
    Returns (candidates, b_prob)
    """
    ns = norm_stats
    model.eval()
    tokens = torch.zeros(1, MAX_BRICKS, FEATURE_DIM)
    mask   = torch.zeros(1, MAX_BRICKS, dtype=torch.bool)
    for i, h in enumerate(history_encoded[:MAX_BRICKS]):
        x, y, z, b, sin_r, cos_r, layer_id, time_index = h
        tokens[0, i] = torch.tensor([
            (x - ns["mean_x"]) / ns["std_x"],
            (y - ns["mean_y"]) / ns["std_y"],
            (z - ns["mean_z"]) / ns["std_z"],
            b, sin_r, cos_r, layer_id / 20.0, time_index / 60.0,
        ])
        mask[0, i] = True
    with torch.no_grad():
        b_logits, pi, mu, sigma = model(tokens.to(device), mask.to(device))
    b_prob = F.softmax(b_logits[0], dim=-1).cpu().numpy()
    pi_np  = pi[0].cpu().numpy()
    mu_np  = mu[0].cpu().numpy()
    sig_np = sigma[0].cpu().numpy()
    candidates = []
    for _ in range(n_candidates):
        k = np.random.choice(len(pi_np), p=pi_np / pi_np.sum())
        s = np.random.randn(5) * sig_np[k] + mu_np[k]
        xr = s[0] * ns["std_x"] + ns["mean_x"]
        yr = s[1] * ns["std_y"] + ns["mean_y"]
        zr = s[2] * ns["std_z"] + ns["mean_z"]
        sin_r, cos_r = s[3], s[4]
        nrm = math.sqrt(sin_r**2 + cos_r**2 + 1e-8)
        sin_r /= nrm; cos_r /= nrm
        r = math.atan2(sin_r, cos_r)
        b = int(np.random.choice(2, p=b_prob))
        candidates.append({"x": xr, "y": yr, "z": zr, "b": b,
                           "sin_r": sin_r, "cos_r": cos_r, "r": r})
    return candidates, b_prob

In [ ]:
# Load best checkpoint and visualise candidates on a validation sample
ckpt = torch.load(CKPT, map_location=device)
model.load_state_dict(ckpt["model_state"])
print(f"Loaded checkpoint: epoch {ckpt['epoch']}, val_loss={ckpt['val_loss']:.4f}")

demo_pair = max(val_pairs, key=lambda p: len(p["history"]))
print(f"Demo seq: {Path(demo_pair['seq_id']).name}  |  history: {len(demo_pair['history'])} bricks")

candidates, b_prob = sample_next_brick(model, demo_pair["history"], norm_stats, n_candidates=100)

fig, ax = plt.subplots(figsize=(7, 7))
hx = [h[0] for h in demo_pair["history"]]
hy = [h[1] for h in demo_pair["history"]]
ax.scatter(hx, hy, c="steelblue", s=40, alpha=0.7, label="History", zorder=3)
tgt = demo_pair["target"]
ax.scatter(tgt["x"], tgt["y"], c="limegreen", s=250, marker="*",
           label="Ground truth", zorder=5)
cx_list = [c["x"] for c in candidates]
cy_list = [c["y"] for c in candidates]
ax.scatter(cx_list, cy_list, c="tomato", s=20, alpha=0.5,
           label=f"Candidates (n={len(candidates)})", zorder=4)
ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)")
ax.set_title("Sampled Next-Brick Candidates -- Top-Down View")
ax.legend(); ax.set_aspect("equal")
plt.tight_layout()
plt.savefig("training_data/candidate_visualization.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"\nBinary probs  --  laying: {b_prob[0]:.3f}  standing: {b_prob[1]:.3f}")
print(f"Ground truth b: {'laying' if tgt['b'] == 0 else 'standing'}")